# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadfarhan2157-source/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Tables: dim_content (metadata, joined on content_hash_id) + fact_content_daily_performance (daily metrics), for month=2026-03.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"SET hf_token='{HF_TOKEN}';")

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("Connected. con and path are ready.")

In [3]:
features_df = con.sql(f"""
    SELECT
      content_hash_id, client_hash_id,
      AVG(impressions) AS avg_impressions_march,
      AVG(clicks) AS avg_clicks_march,
      AVG(avg_position) AS avg_position_march,
      SUM(clicks) / NULLIF(SUM(impressions), 0) AS ctr_march,
      COUNT(*) AS days_with_data
    FROM '{path}'
    GROUP BY 1, 2
""").df()
features_df.head()


one line per feature, exactly what the brief requires:

avg_impressions_march  available at decision time: purely historical.
avg_clicks_march  available at decision time: purely historical.
avg_position_march  available at decision time: a past ranking measurement.
ctr_march  available at decision time: derived only from past clicks/impressions.
days_with_data — available at decision time: known immediately from coverage so far.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# THE TRAP — deliberately add a label-derived column and watch the score jump
april_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

leak_check = con.sql(f"""
    WITH march AS (SELECT content_hash_id, client_hash_id, AVG(clicks) AS avg_clicks_march
                   FROM '{path}' GROUP BY 1,2),
         april AS (SELECT content_hash_id, client_hash_id, AVG(clicks) AS avg_clicks_april
                   FROM '{april_path}' GROUP BY 1,2)
    SELECT m.*, a.avg_clicks_april,
           (a.avg_clicks_april < m.avg_clicks_march) AS leaked_future_decline_label
    FROM march m JOIN april a USING (content_hash_id, client_hash_id)
""").df()

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = leak_check["leaked_future_decline_label"]

X_leaky = leak_check[["avg_clicks_march", "leaked_future_decline_label"]]
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.3, random_state=1)
print("AUC WITH leak:", roc_auc_score(yte, LogisticRegression().fit(Xtr, ytr).predict_proba(Xte)[:,1]))

X_honest = leak_check[["avg_clicks_march"]]
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=1)
print("AUC WITHOUT leak (honest):", roc_auc_score(yte, LogisticRegression().fit(Xtr, ytr).predict_proba(Xte)[:,1]))


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell you about seasonality  one month, and only 9 of 70 clients have 12+ months of history at any point per the lane guide. It also can't be trusted for clients whose gsc_data_start/ga4_data_start falls partway through March — their row count for the month understates a full month of tracking, and rows before that start date reflect "no tracking yet," not "no traffic."

In [ ]:
limits_check = con.sql(f"""
    SELECT client_hash_id, COUNT(*) AS march_rows
    FROM '{path}'
    GROUP BY 1
    ORDER BY march_rows ASC
    LIMIT 10
""").df()
print("Clients with fewest March rows (likely partial-month tracking):")
limits_check


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.